# AI-Based Behavioral IDS - Evaluation & Visualization

This notebook provides comprehensive evaluation and visualization of the IDS model performance.

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from evaluation.metrics import evaluate_model, compute_ids_specific_metrics
from evaluation.visualization import (
    plot_confusion_matrix, plot_roc_curve, plot_precision_recall_curve,
    plot_score_distribution, plot_model_comparison
)
from evaluation.ablation_study import AblationStudy

plt.style.use('seaborn-v0_8-darkgrid')
print("Setup complete!")

## 1. Generate Evaluation Data

Using synthetic data for demonstration.

In [ ]:
# Generate sample predictions
np.random.seed(42)
n_samples = 2000

# True labels (85% normal, 15% attack)
y_true = np.random.choice([0, 1], size=n_samples, p=[0.85, 0.15])

# Simulated predictions (high accuracy)
y_pred = y_true.copy()
noise = np.random.random(n_samples) < 0.02  # 2% error rate
y_pred[noise] = 1 - y_pred[noise]

# Simulated scores
y_scores = np.where(y_true == 1, 
                   np.random.beta(8, 2, n_samples),
                   np.random.beta(2, 8, n_samples))

print(f"Samples: {n_samples}")
print(f"Normal: {np.sum(y_true==0)}, Attack: {np.sum(y_true==1)}")
print(f"Accuracy: {np.mean(y_true == y_pred):.4f}")

## 2. Comprehensive Evaluation

In [ ]:
# Run full evaluation
results = evaluate_model(y_true, y_pred, y_scores, model_name="Hybrid IDS")

# IDS-specific metrics
ids_metrics = compute_ids_specific_metrics(y_true, y_pred)

print("\nIDS-Specific Metrics:")
for k, v in ids_metrics.items():
    print(f"  {k}: {v:.4f}")

## 3. Confusion Matrix

In [ ]:
fig = plot_confusion_matrix(
    y_true, y_pred, 
    class_names=['Normal', 'Attack'],
    model_name="Hybrid IDS"
)
plt.show()

## 4. ROC Curve

In [ ]:
fig = plot_roc_curve(y_true, y_scores, model_name="Hybrid IDS")
plt.show()

## 5. Precision-Recall Curve

In [ ]:
fig = plot_precision_recall_curve(y_true, y_scores, model_name="Hybrid IDS")
plt.show()

## 6. Score Distribution

In [ ]:
fig = plot_score_distribution(
    y_true, y_scores, 
    threshold=0.5,
    model_name="Hybrid IDS"
)
plt.show()

## 7. Model Comparison

In [ ]:
# Simulate multiple model results
model_results = {
    'LSTM Only': {'accuracy': 0.9254, 'precision': 0.89, 'recall': 0.92, 'f1_score': 0.9053},
    'Isolation Forest': {'accuracy': 0.9905, 'precision': 0.99, 'recall': 0.99, 'f1_score': 0.9941},
    'DNN Only': {'accuracy': 0.9945, 'precision': 0.99, 'recall': 0.99, 'f1_score': 0.9893},
    'Hybrid (Full)': {'accuracy': 0.9966, 'precision': 0.996, 'recall': 0.996, 'f1_score': 0.9927}
}

fig = plot_model_comparison(model_results, metric='f1_score')
plt.show()

## 8. Summary Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Confusion Matrix
cm = np.array([[results['confusion_matrix']['tn'], results['confusion_matrix']['fp']],
               [results['confusion_matrix']['fn'], results['confusion_matrix']['tp']]])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0,0],
          xticklabels=['Normal', 'Attack'], yticklabels=['Normal', 'Attack'])
axes[0,0].set_title('Confusion Matrix', fontweight='bold')

# 2. Metrics Bar Chart
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'Specificity']
metrics_values = [
    ids_metrics['detection_rate'],
    ids_metrics['positive_predictive_value'],
    ids_metrics['detection_rate'],
    ids_metrics['f_measure'],
    ids_metrics['specificity']
]
axes[0,1].bar(metrics_names, metrics_values, color='steelblue', edgecolor='black')
axes[0,1].set_ylim(0, 1.1)
axes[0,1].set_title('Key Metrics', fontweight='bold')
axes[0,1].set_ylabel('Score')
for i, v in enumerate(metrics_values):
    axes[0,1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# 3. Score Distribution
axes[1,0].hist(y_scores[y_true==0], bins=50, alpha=0.6, label='Normal', density=True)
axes[1,0].hist(y_scores[y_true==1], bins=50, alpha=0.6, label='Attack', density=True)
axes[1,0].set_title('Anomaly Score Distribution', fontweight='bold')
axes[1,0].legend()

# 4. Model Comparison
model_names = list(model_results.keys())
f1_values = [model_results[m]['f1_score'] for m in model_names]
axes[1,1].barh(model_names, f1_values, color='coral', edgecolor='black')
axes[1,1].set_xlim(0.8, 1.0)
axes[1,1].set_title('F1 Score Comparison', fontweight='bold')
axes[1,1].set_xlabel('F1 Score')

plt.suptitle('IDS Evaluation Dashboard', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nEvaluation complete!")